# Revisão do M8

Este notebook audita o M8 e define medidas de retrabalho limpo adequadas à RQ3.

**Conclusão:** a razão legada é reproduzível, mas inclui uma política de exclusão mais fraca que M4 e colapsa magnitude, elegibilidade e dinâmica temporal. A proposta usa somente source/test, separa magnitude de proporção e mostra a trajetória móvel do retrabalho tardio.

## Estrutura recomendada

| Saída | Pergunta respondida | Relação com outras métricas |
|---|---|---|
| **M8a - Magnitude de retrabalho limpo** | Quantas linhas source/test pré-existentes foram alteradas no T3? | M4 mede toda mudança limpa; M8a isola somente caminhos pré-existentes |
| **M8b - Participação de retrabalho limpo** | Que fração do churn limpo T3 alterou caminhos pré-existentes? | Condicionada a baseline elegível; não confunde escala com proporção |
| **M8c - Trajetória móvel de retrabalho limpo** | O retrabalho se acumula ou é localizado perto da apresentação? | Complementa M4d com proveniência dos caminhos |
| **M8d - Elegibilidade de baseline** | Havia caminhos limpos T1/T2 que poderiam ser revisados? | Evita interpretar zero como estabilidade quando não havia baseline |

M3 mede autoria/tempo, M4 magnitude de toda mudança limpa e M7 inatividade. M8 usa origem do caminho para medir somente revisão de trabalho previamente versionado.

## 1. Vínculo com a RQ3

O notebook extrai a vinculação de M8 diretamente do texto atual do paper e interrompe a execução se a métrica não pertencer à RQ3.

In [ ]:
from hashlib import sha256
from pathlib import Path
import importlib.util
import re

import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'paper_v8/latex_code/main.tex').is_file():
            return candidate
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = find_project_root(Path.cwd())
PAPER_PATH = PROJECT_ROOT / 'paper_v8/latex_code/main.tex'
M8_PATH = PROJECT_ROOT / 'paper_v8/data/m8_rework_severity_ratio.csv'
SIGNALS_PATH = PROJECT_ROOT / 'paper_v4/advanced_metrics/outputs/team_level_signals.csv'
FILES_PATH = PROJECT_ROOT / 'data/lake/git_files.parquet'

paper_text = PAPER_PATH.read_text(encoding='utf-8')
rq_matches = dict(re.findall(r'\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n', paper_text))
m8_position = paper_text.index(r'\textbf{M8 --')
rq3_position = paper_text.index(r'\subsubsection{RQ3:')
detected_rq = 'RQ3' if rq3_position < m8_position else 'UNKNOWN'
assert detected_rq == 'RQ3'
assert rq_matches[detected_rq]
pd.DataFrame([{'metric':'M8','detected_rq':detected_rq,'rq_text':rq_matches[detected_rq],'paper_sha256':sha256(paper_text.encode()).hexdigest()}])

In [ ]:
CONFIG = {
    'metric': 'M8',
    'expected_rq': 'RQ3',
    'unit_of_analysis': 'team_semester_then_rolling_window',
    'source_contract': 'data/lake/git_files.parquet',
    'primary_outputs': ['clean_rework_churn_t3', 'clean_rework_ratio_t3', 'baseline_eligible_for_rework_t3', 'rolling_7d_clean_rework_churn'],
    'included_categories': ['source', 'test'],
    'rolling_window_days': 7,
    'rolling_step_days': 1,
    'rolling_end_day_range_relative_to_t3': [-21, 7],
    'timestamp_field': 'committer_date',
    'inference': 'exploratory_descriptive',
    'export_outputs': False,
}
assert CONFIG['expected_rq'] == detected_rq
assert CONFIG['rolling_window_days'] == 7
CONFIG

## 2. Auditoria do M8 legado

O M8 publicado reexpõe `rework_churn_t3` e `deferred_churn_t3`, calcula o total e a razão. A auditoria reproduz a aritmética; isso não valida a classificação semântica de toda mudança em caminho pré-existente como retrabalho destrutivo.

In [ ]:
legacy_m8 = pd.read_csv(M8_PATH, dtype={'Semestre':str})
signals = pd.read_csv(SIGNALS_PATH, dtype={'Semestre':str})
recalculated = signals[['ID_Equipe','Semestre','rework_churn_t3','deferred_churn_t3']].copy()
recalculated['total_churn_t3'] = recalculated.rework_churn_t3 + recalculated.deferred_churn_t3
recalculated['rework_ratio_t3'] = recalculated.rework_churn_t3 / recalculated.total_churn_t3.where(recalculated.total_churn_t3.ne(0))
legacy_comparison = legacy_m8.merge(recalculated, on=['ID_Equipe','Semestre'], suffixes=('_published','_recalculated'), validate='one_to_one')
for column in ['rework_churn_t3','deferred_churn_t3','total_churn_t3','rework_ratio_t3']:
    assert np.allclose(legacy_comparison[f'{column}_published'], legacy_comparison[f'{column}_recalculated'], equal_nan=True)
legacy_audit = pd.DataFrame([('team_semester_n',len(legacy_m8)),('zero_total_n',int(legacy_m8.total_churn_t3.eq(0).sum())),('max_legacy_rework',float(legacy_m8.rework_churn_t3.max()))],columns=['check','value'])
legacy_audit

## 3. Proveniência limpa de caminhos

A classificação reutiliza o contrato M4: categoria `source`/`test` e extensão na whitelist. Para cada caminho limpo, a primeira janela observada define a baseline: T1/T2 torna o caminho elegível para retrabalho no T3; primeira aparição T3 é desenvolvimento diferido. Isso continua sendo uma proxy de proveniência, não uma classificação semântica de defeito.

In [ ]:
if str(PROJECT_ROOT) not in __import__('sys').path:
    __import__('sys').path.insert(0, str(PROJECT_ROOT))
from pipeline_config import (
    FILE_CATEGORY_DEFINITION_VERSION,
    SOURCE_CODE_EXTENSION_ALLOWLIST,
    EVALUATOR_TEMPORAL_CUTS,
    is_measurement_code_path,
)

engine_spec = importlib.util.spec_from_file_location('cross_evidence_engine', PROJECT_ROOT / '08_cross_evidence_engine.py')
assert engine_spec and engine_spec.loader
engine = importlib.util.module_from_spec(engine_spec)
engine_spec.loader.exec_module(engine)
classify_file_category = engine.classify_file_category

files = pd.read_parquet(FILES_PATH).copy()
required_file_columns = {'ID_Equipe','Semestre','timestamp','temporal_marker','file_path','file_extension','lines_added','lines_deleted'}
assert not (required_file_columns - set(files))
files['file_category'] = [classify_file_category(row.file_path, row.file_extension, row.change_status, row.file_path_old)['file_category'] for row in files.itertuples(index=False)]
files['included_in_m8'] = [
    category in CONFIG['included_categories'] and is_measurement_code_path(path)
    for path, category in zip(files['file_path'], files['file_category'])
]
files['churn_lines'] = files.lines_added.fillna(0) + files.lines_deleted.fillna(0)
files['marker_rank'] = files.temporal_marker.map({'T1':1,'T2':2,'T3':3})
clean_origins = (files.loc[files.included_in_m8].groupby(['ID_Equipe','Semestre','file_path'],as_index=False).marker_rank.min().rename(columns={'marker_rank':'first_clean_marker_rank'}))
files = files.merge(clean_origins, on=['ID_Equipe','Semestre','file_path'], how='left', validate='many_to_one')
files['clean_origin_type'] = np.select([files.first_clean_marker_rank.isin([1,2]),files.first_clean_marker_rank.eq(3)],['rework','deferred'],default='not_included')
history_paths = files.file_path.astype(str).str.startswith('.history/')
backup_paths = files.file_path.astype(str).str.contains(r'(?:^|/)backups?/', regex=True)
assert not files.loc[history_paths, 'included_in_m8'].any()
assert files.loc[history_paths, 'file_category'].eq('generated').all()
assert not files.loc[backup_paths, 'included_in_m8'].any()
assert files.loc[backup_paths, 'file_category'].eq('generated').all()
assert files.loc[files.included_in_m8, 'file_extension'].fillna('').str.lower().isin(SOURCE_CODE_EXTENSION_ALLOWLIST).all()
assert not files.loc[files.included_in_m8, 'first_clean_marker_rank'].isna().any()
files[['file_category','included_in_m8','clean_origin_type']].value_counts().rename('event_n').reset_index()

In [ ]:
team_universe = legacy_m8[['ID_Equipe','Semestre']].copy()
t3_clean = files.loc[files.temporal_marker.eq('T3') & files.included_in_m8].copy()
clean_t3 = (t3_clean.groupby(['ID_Equipe','Semestre','clean_origin_type'],as_index=False).churn_lines.sum().pivot_table(index=['ID_Equipe','Semestre'],columns='clean_origin_type',values='churn_lines',fill_value=0).reset_index())
for column in ['rework','deferred']:
    if column not in clean_t3:
        clean_t3[column] = 0.0
clean_m8 = team_universe.merge(clean_t3[['ID_Equipe','Semestre','rework','deferred']],on=['ID_Equipe','Semestre'],how='left').fillna({'rework':0.0,'deferred':0.0})
clean_m8 = clean_m8.rename(columns={'rework':'clean_rework_churn_t3','deferred':'clean_deferred_churn_t3'})
clean_m8['clean_total_churn_t3'] = clean_m8.clean_rework_churn_t3 + clean_m8.clean_deferred_churn_t3
clean_m8['clean_rework_ratio_t3'] = clean_m8.clean_rework_churn_t3 / clean_m8.clean_total_churn_t3.where(clean_m8.clean_total_churn_t3.ne(0))
prior_paths = clean_origins.loc[clean_origins.first_clean_marker_rank.isin([1,2])].groupby(['ID_Equipe','Semestre']).size().rename('prior_clean_path_n').reset_index()
clean_m8 = clean_m8.merge(prior_paths,on=['ID_Equipe','Semestre'],how='left').fillna({'prior_clean_path_n':0})
clean_m8['baseline_eligible_for_rework_t3'] = clean_m8.prior_clean_path_n.gt(0)
assert len(clean_m8) == 14
assert clean_m8.loc[~clean_m8.baseline_eligible_for_rework_t3,'clean_rework_churn_t3'].eq(0).all()
clean_m8.sort_values(['Semestre','ID_Equipe'])

## 4. Sensibilidade ao filtro e não redundância

A comparação abaixo mede o efeito de aplicar o contrato limpo de M4. M8 não é redundante com M4d: M4d inclui toda mudança limpa; M8 inclui somente mudanças em caminhos com baseline T1/T2. Não é redundante com M6/M7: elegibilidade de baseline é derivada dos caminhos, não uma nota ou ausência de planejamento.

In [ ]:
filter_sensitivity = legacy_m8.merge(clean_m8,on=['ID_Equipe','Semestre'],validate='one_to_one')
filter_sensitivity['legacy_minus_clean_rework'] = filter_sensitivity.rework_churn_t3 - filter_sensitivity.clean_rework_churn_t3
filter_sensitivity['legacy_minus_clean_deferred'] = filter_sensitivity.deferred_churn_t3 - filter_sensitivity.clean_deferred_churn_t3
filter_sensitivity['ratio_difference'] = filter_sensitivity.rework_ratio_t3 - filter_sensitivity.clean_rework_ratio_t3
assert filter_sensitivity.clean_rework_churn_t3.le(filter_sensitivity.rework_churn_t3).all()
filter_sensitivity.sort_values('ratio_difference', key=lambda values: values.abs(), ascending=False)[['ID_Equipe','Semestre','rework_ratio_t3','clean_rework_ratio_t3','ratio_difference','baseline_eligible_for_rework_t3']]

## 5. Trajetória móvel de retrabalho limpo

A série usa o último voto T3 de cada equipe como âncora e janelas retrospectivas de 7 dias, de 21 dias antes a 7 dias depois. Somente eventos T3 em caminhos limpos com baseline T1/T2 contribuem para M8c. A janela posterior é mostrada para revelar se o pico é localizado, mas não deve ser tratada como desfecho pré-apresentação.

In [ ]:
def last_t3_vote_anchors(project_root: Path) -> pd.DataFrame:
    rows=[]
    for semester, ranges in EVALUATOR_TEMPORAL_CUTS.items():
        votes=pd.read_csv(project_root / f'data/processed/forms/{semester}/avaliadores.csv')
        votes['ID_Equipe']=(votes['To which group do these scores refer?'].str.extract(r'Group\s+(\d+)',expand=False).astype(int).map(lambda value:f'TEAM_{value:02d}'))
        votes['vote_at']=pd.to_datetime(votes['Timestamp'],format='mixed').dt.tz_localize('America/Fortaleza')
        start,end=ranges['T3']
        t3=votes.loc[votes.vote_at.dt.date.between(pd.Timestamp(start).date(),pd.Timestamp(end).date())]
        rows.append(t3.groupby('ID_Equipe',as_index=False).vote_at.max().assign(Semestre=semester))
    return pd.concat(rows,ignore_index=True)

t3_anchors=last_t3_vote_anchors(PROJECT_ROOT)
rolling_rework_rows=[]
for anchor in t3_anchors.to_dict('records'):
    team_events=t3_clean.loc[t3_clean.Semestre.astype(str).eq(anchor['Semestre']) & t3_clean.ID_Equipe.eq(anchor['ID_Equipe'])].copy()
    team_events=team_events.loc[team_events.clean_origin_type.eq('rework')]
    t3_anchor=anchor['vote_at'].tz_convert('UTC')
    for day_offset in range(CONFIG['rolling_end_day_range_relative_to_t3'][0],CONFIG['rolling_end_day_range_relative_to_t3'][1]+1,CONFIG['rolling_step_days']):
        window_end=t3_anchor+pd.Timedelta(days=day_offset)
        window_start=window_end-pd.Timedelta(days=CONFIG['rolling_window_days'])
        current=team_events.loc[team_events.timestamp.ge(window_start) & team_events.timestamp.lt(window_end)]
        rolling_rework_rows.append({'ID_Equipe':anchor['ID_Equipe'],'Semestre':anchor['Semestre'],'window_end_day_relative_to_t3':day_offset,'clean_rework_churn_7d':float(current.churn_lines.sum()),'reworked_clean_path_n_7d':int(current.file_path.nunique())})
rolling_rework=pd.DataFrame(rolling_rework_rows)
rolling_rework_cohort=(rolling_rework.groupby(['Semestre','window_end_day_relative_to_t3'],as_index=False).agg(team_n=('ID_Equipe','size'),total_clean_rework_churn_7d=('clean_rework_churn_7d','sum'),median_clean_rework_churn_7d=('clean_rework_churn_7d','median'),teams_with_clean_rework_n=('clean_rework_churn_7d',lambda values:int(values.gt(0).sum()))))
rolling_rework_cohort['teams_with_clean_rework_share']=rolling_rework_cohort.teams_with_clean_rework_n/rolling_rework_cohort.team_n
assert len(rolling_rework)==14*29
assert rolling_rework.groupby(['ID_Equipe','Semestre']).size().eq(29).all()
rolling_rework_cohort.loc[rolling_rework_cohort.window_end_day_relative_to_t3.isin([-21,-14,-7,0,7])].sort_values(['Semestre','window_end_day_relative_to_t3'])

### 5.1 Diagnóstico da distribuição entre equipes

Total e mediana respondem perguntas diferentes. O total mostra a carga agregada; a mediana descreve a equipe típica, incluindo equipes sem retrabalho naquela janela. Toda leitura de pico deve reportar também quantas equipes contribuíram e o maior valor individual.

In [ ]:
rolling_rework_distribution = (
    rolling_rework.groupby(["Semestre", "window_end_day_relative_to_t3"], as_index=False)
    .agg(
        team_n=("ID_Equipe", "size"),
        zero_rework_team_n=("clean_rework_churn_7d", lambda values: int(values.eq(0).sum())),
        positive_rework_team_n=("clean_rework_churn_7d", lambda values: int(values.gt(0).sum())),
        total_clean_rework_churn_7d=("clean_rework_churn_7d", "sum"),
        median_clean_rework_churn_7d=("clean_rework_churn_7d", "median"),
        mean_clean_rework_churn_7d=("clean_rework_churn_7d", "mean"),
        max_clean_rework_churn_7d=("clean_rework_churn_7d", "max"),
    )
)
rolling_rework_distribution["max_share_of_total"] = (
    rolling_rework_distribution["max_clean_rework_churn_7d"]
    / rolling_rework_distribution["total_clean_rework_churn_7d"].replace(0, np.nan)
)
assert (
    rolling_rework_distribution["zero_rework_team_n"]
    + rolling_rework_distribution["positive_rework_team_n"]
).eq(rolling_rework_distribution["team_n"]).all()
assert rolling_rework_distribution["median_clean_rework_churn_7d"].ge(0).all()
peak_distribution = rolling_rework_distribution.loc[
    rolling_rework_distribution.groupby("Semestre")["total_clean_rework_churn_7d"].idxmax()
].sort_values("Semestre")
peak_distribution

## 6. Decisão e correções do paper

**Decisão:** substituir M8 legado por M8a--M8d. Reportar magnitude e razão limpas lado a lado, sempre com elegibilidade de baseline. Descrever M8c como dinâmica de mudanças em caminhos pré-existentes, não como diagnóstico de defeito. M9 deve testar M8a e M8b separadamente, com as equipes não elegíveis reportadas como estrato distinto, sem imputar zero como estabilidade.

In [ ]:
m8_decision=pd.DataFrame([
 ('M8a','adopt','clean rework churn in source/test paths first observed at T1/T2'),
 ('M8b','adopt conditionally','clean rework ratio; interpret only with nonzero clean T3 churn and baseline eligibility'),
 ('M8c','adopt descriptively','7-day rolling clean rework during late stage; no causal or inferential claim'),
 ('M8d','adopt','prior clean path count and eligibility shown alongside zero rework'),
 ('M9','revise','analyze M8a and M8b separately; do not use a single rework score or impute ineligible teams'),
],columns=['component','decision','reason'])
assert CONFIG['inference']=='exploratory_descriptive'
assert clean_m8.baseline_eligible_for_rework_t3.any()
m8_decision

In [ ]:
assert detected_rq=='RQ3'
assert legacy_m8.duplicated(['ID_Equipe','Semestre']).sum()==0
assert len(clean_m8)==14
assert clean_m8.loc[~clean_m8.baseline_eligible_for_rework_t3,'clean_rework_churn_t3'].eq(0).all()
assert len(rolling_rework)==14*29
assert rolling_rework.clean_rework_churn_7d.ge(0).all()
assert not files.loc[files.file_path.astype(str).str.startswith('.history/'),'included_in_m8'].any()
m8_evidence_manifest={'metric':'M8','rq':detected_rq,'analysis_level':CONFIG['unit_of_analysis'],'source_contract':str(FILES_PATH.relative_to(PROJECT_ROOT)),'included_categories':CONFIG['included_categories'],'extension_allowlist':'SOURCE_CODE_EXTENSION_ALLOWLIST','artifact_policy_version':FILE_CATEGORY_DEFINITION_VERSION,'rolling_window_days':CONFIG['rolling_window_days'],'rolling_end_day_range_relative_to_t3':CONFIG['rolling_end_day_range_relative_to_t3'],'team_semester_n':int(len(clean_m8)),'daily_windows_n':int(len(rolling_rework)),'inference':CONFIG['inference'],'construct_limit':'file_provenance_proxy_not_semantic_destructive_rework'}
m8_evidence_manifest